### Imports

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import pickle

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from tqdm import tqdm

### Data Preprocessing

In [2]:
# DATA_PATH = "../../../data/processed/ratings_clean.csv"
DATA_PATH = "../../../data/processed/ratings_clean.csv"

df = pd.read_csv(DATA_PATH)
df = df[[
    'user_id',
    'item_id',
    'rating',
    'timestamp'
]]
print(df.head())

   user_id  item_id  rating            timestamp
0  1489087    81117     5.0  1997-12-31 15:29:20
1  2344135   163427     5.0  1998-02-05 16:06:46
2  1295441    88640     5.0  1998-03-28 22:41:56
3  3060623   237831     4.0  1998-05-18 22:38:54
4  1787647    66994     5.0  1998-06-13 15:57:44


In [3]:
df = df.sort_values(
    by='timestamp'
)
print(df.head())

   user_id  item_id  rating            timestamp
0  1489087    81117     5.0  1997-12-31 15:29:20
1  2344135   163427     5.0  1998-02-05 16:06:46
2  1295441    88640     5.0  1998-03-28 22:41:56
3  3060623   237831     4.0  1998-05-18 22:38:54
4  1787647    66994     5.0  1998-06-13 15:57:44


In [4]:
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
num_ratings = len(df)

print(f"Users   : {num_users}")
print(f"Items   : {num_items}")
print(f"Ratings : {num_ratings}")

Users   : 239616
Items   : 99223
Ratings : 4628190


In [5]:
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
num_ratings = len(df)

print(f"Users   : {num_users}")
print(f"Items   : {num_items}")
print(f"Ratings : {num_ratings}")

Users   : 239616
Items   : 99223
Ratings : 4628190


In [6]:
user_ids = df['user_id'].unique()
item_ids = df['item_id'].unique()


user_to_index = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}


item_to_index = {
    item_id: idx + 1
    for idx, item_id in enumerate(item_ids)
}


index_to_user = {
    idx: user_id
    for user_id, idx in user_to_index.items()
}


index_to_item = {
    idx: item_id
    for item_id, idx in item_to_index.items()
}


# Create encoded columns

df['user_idx'] = df['user_id'].map(user_to_index)
df['item_idx'] = df['item_id'].map(item_to_index)


print(df.head())

   user_id  item_id  rating            timestamp  user_idx  item_idx
0  1489087    81117     5.0  1997-12-31 15:29:20         0         1
1  2344135   163427     5.0  1998-02-05 16:06:46         1         2
2  1295441    88640     5.0  1998-03-28 22:41:56         2         3
3  3060623   237831     4.0  1998-05-18 22:38:54         3         4
4  1787647    66994     5.0  1998-06-13 15:57:44         4         5


In [7]:
from collections import defaultdict

user_histories = defaultdict(list)

In [8]:
for row in df.itertuples():

    user_histories[row.user_idx].append(
        (
            row.item_idx,
            row.rating
        )
    )

In [9]:
user_histories[10]

[(12, 2.0),
 (1558, 4.0),
 (2242, 4.0),
 (3065, 4.0),
 (2088, 3.0),
 (18068, 4.0),
 (17743, 5.0),
 (8445, 3.0),
 (22238, 5.0),
 (17007, 2.0),
 (2070, 2.0)]

In [10]:
df = df.drop_duplicates(
    subset=['user_id', 'item_id'],
    keep='last'
)

In [11]:
train_histories = {}
val_histories = {}
test_histories = {}

for user_id, interactions in user_histories.items():

    if len(interactions) < 3:
        continue

    train_histories[user_id] = interactions[:-2]

    val_histories[user_id] = interactions[:-1]

    test_histories[user_id] = interactions

In [12]:
with open("../../artifacts/index_to_item.pkl", "wb") as f:
    pickle.dump(index_to_item, f)

with open("../../artifacts/item_to_index.pkl", "wb") as f:
    pickle.dump(item_to_index, f)

with open("../../artifacts/user_histories.pkl", "wb") as f:
    pickle.dump(user_histories, f)

### Dynamic User Embedding


In [27]:
class DynamicUserDataset(Dataset):

    def __init__(
        self,
        user_histories,
        max_history=50
    ):

        self.samples = []

        self.max_history = max_history

        for user_id, interactions in user_histories.items():

            if len(interactions) < 2:
                continue

            for i in range(1, len(interactions)):

                history = interactions[:i]

                target_item, target_rating = (
                    interactions[i]
                )

                history_items = [
                    x[0]
                    for x in history
                ]

                history_ratings = [
                    x[1]
                    for x in history
                ]

                # Truncate history
                history_items = history_items[
                    -max_history:
                ]

                history_ratings = history_ratings[
                    -max_history:
                ]

                self.samples.append(
                    (
                        history_items,
                        history_ratings,
                        target_item,
                        target_rating
                    )
                )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        return self.samples[idx]

In [28]:
def collate_fn(batch):

    histories = []
    ratings = []
    targets = []
    labels = []

    max_len = max(
        len(x[0])
        for x in batch
    )

    for history, history_ratings, target, label in batch:

        pad_len = max_len - len(history)

        histories.append(
            history + [0] * pad_len
        )

        ratings.append(
            history_ratings + [0] * pad_len
        )

        targets.append(target)

        labels.append(label)

    return (

        torch.tensor(
            histories,
            dtype=torch.long
        ),

        torch.tensor(
            ratings,
            dtype=torch.float32
        ),

        torch.tensor(
            targets,
            dtype=torch.long
        ),

        torch.tensor(
            labels,
            dtype=torch.float32
        )
    )

In [39]:
class DynamicNCF(nn.Module):
    def __init__(self, num_items, embedding_dim=32):
        super().__init__()

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim,
            padding_idx=0
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def build_user_embedding(self, history_items, history_ratings):
        """
        Called ONCE to initialize embedding from history.
        history_items:   (B, seq_len) long tensor
        history_ratings: (B, seq_len) float tensor
        Returns: user_embedding (B, dim), weight_sum (B, 1)
        """
        history_embeds = self.item_embedding(history_items)          # (B, seq, dim)
        # mask = (history_items != 0).unsqueeze(-1)
        weights  = (history_ratings).unsqueeze(-1)               # (B, seq, 1)
        # weights = weights * mask
        weight_sum     = history_ratings.sum(dim=1, keepdim=True)    # (B, 1)
        # abs_weight_sum = (
        #     weight.abs().sum(dim=1, keepdim=True)
        # )
        
        mask           = (history_items != 0).unsqueeze(-1)          # (B, seq, 1)
        weighted       = history_embeds * weights * mask             # (B, seq, dim)

        user_embedding = weighted.sum(dim=1) / weight_sum.clamp(min=1e-6)  # (B, dim)

        return user_embedding, weight_sum

    # def build_user_embedding(
    #     self,
    #     history_items,
    #     history_ratings
    # ):
    
    #     history_embeds = self.item_embedding(
    #         history_items
    #     )
    
    #     mask = (
    #         history_items != 0
    #     ).unsqueeze(-1)
    
    #     weights = (
    #         history_ratings - 3.0
    #     ).unsqueeze(-1)
    
    #     weights = weights * mask
    
    #     weighted_embeds = (
    #         history_embeds * weights
    #     )
    
    #     embedding_sum = (
    #         weighted_embeds.sum(dim=1)
    #     )
    
    #     weight_sum = (
    #         weights.abs()
    #         .sum(dim=1)
    #     )
    
    #     user_embedding = (
    #         embedding_sum
    #         /
    #         weight_sum.clamp(min=1e-6)
    #     )
    
    #     return (
    #         user_embedding,
    #         embedding_sum,
    #         weight_sum
    #     )

    def update_user_embedding(self, user_embedding, weight_sum, new_item, new_rating):
        """
        Incrementally update a single user's embedding.
        user_embedding: (1, dim)
        weight_sum:     scalar or (1, 1)
        new_item:       (1,) long tensor
        new_rating:     float
        Returns: updated user_embedding (1, dim), updated weight_sum
        """
        new_item_embed = self.item_embedding(new_item)               # (1, dim)
        new_weight_sum = weight_sum + new_rating

        user_embedding = (
            user_embedding * weight_sum + new_item_embed * new_rating
        ) / new_weight_sum.clamp(min=1e-6)

        return user_embedding, new_weight_sum

    # def update_user_embedding(
    #     self,
    #     embedding_sum,
    #     weight_sum,
    #     new_item,
    #     new_rating
    # ):
    
    #     new_item_embed = self.item_embedding(
    #         new_item
    #     )
    
    #     weight = new_rating - 3.0
    
    #     embedding_sum = (
    #         embedding_sum
    #         +
    #         new_item_embed * weight
    #     )
    
    #     weight_sum = (
    #         weight_sum
    #         +
    #         abs(weight)
    #     )
    
    #     user_embedding = (
    #         embedding_sum
    #         /
    #         weight_sum.clamp(min=1e-6)
    #     )
    
    #     return (
    #         user_embedding,
    #         embedding_sum,
    #         weight_sum
    #     )

    def forward(self, user_embedding, target_items):
        """
        user_embedding: (B, dim) — pre-built, passed in
        target_items:   (B,)    — item indices
        """
        target_embedding = self.item_embedding(target_items)         # (B, dim)
        x = torch.cat([user_embedding, target_embedding], dim=1)
        return 5.0 * torch.sigmoid(self.mlp(x).squeeze())

In [35]:
train_dataset = DynamicUserDataset(
    train_histories
)

val_dataset = DynamicUserDataset(
    val_histories
)

test_dataset = DynamicUserDataset(
    test_histories
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4096,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4096,
    shuffle=False,
    collate_fn=collate_fn
)

In [40]:
num_items = len(item_to_index) + 1
model = DynamicNCF(
    num_items=num_items,
    embedding_dim=32
)

print(model)

DynamicNCF(
  (item_embedding): Embedding(99224, 32, padding_idx=0)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [41]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

model = model.to(DEVICE)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

EPOCHS = 10

PATIENCE = 3

best_val_loss = float('inf')

patience_counter = 0

lambda_reg = 1e-6

cuda


### Latest Training Loop

In [42]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for (
        history_items, 
        history_ratings, 
        target_items, 
        labels) in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        history_items   = history_items.to(DEVICE)
        history_ratings = history_ratings.to(DEVICE)
        target_items    = target_items.to(DEVICE)
        labels          = labels.to(DEVICE)


        # Build User Embedding from history
        user_embedding, _ = model.build_user_embedding(history_items, history_ratings)
        predictions = model(user_embedding, target_items)

        loss = criterion(predictions, labels)
        reg_loss = model.item_embedding.weight.norm(2).pow(2)
        loss = loss + lambda_reg * reg_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0

    with torch.inference_mode():
        for (
            history_items,
            history_ratings,
            target_items,
            labels
        ) in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}"):

            history_items   = history_items.to(DEVICE)
            history_ratings = history_ratings.to(DEVICE)
            target_items    = target_items.to(DEVICE)
            labels          = labels.to(DEVICE)

            user_embedding, _ = model.build_user_embedding(history_items, history_ratings)

            predictions = model(user_embedding, target_items)

            loss     = criterion(predictions, labels)
            reg_loss = model.item_embedding.weight.norm(2).pow(2)
            loss     = loss + lambda_reg * reg_loss
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_dynamic_ncf_online_debug.pth")
        print("Validation improved → model saved")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered")
            break
            

Validation Epoch 1: 100%|██████████| 1013/1013 [00:47<00:00, 21.49it/s]


Epoch 1 | Train Loss: 3.1177 | Val Loss: 2.0974
Validation improved → model saved


Validation Epoch 2: 100%|██████████| 1013/1013 [00:54<00:00, 18.51it/s]


Epoch 2 | Train Loss: 1.6761 | Val Loss: 1.3806
Validation improved → model saved


Validation Epoch 3: 100%|██████████| 1013/1013 [00:53<00:00, 18.88it/s]


...

Epoch 8 | Train Loss: 0.9717 | Val Loss: 0.9091
Validation improved → model saved


Validation Epoch 9: 100%|██████████| 1013/1013 [00:45<00:00, 22.29it/s]


Epoch 9 | Train Loss: 0.9369 | Val Loss: 0.8716
Validation improved → model saved


Validation Epoch 10: 100%|██████████| 1013/1013 [00:45<00:00, 22.44it/s]

Epoch 10 | Train Loss: 0.9034 | Val Loss: 0.8356
Validation improved → model saved


In [43]:
def recommend_movies(
    user_id,
    top_k=10
):

    model.eval()

    # =====================================================
    # GET USER HISTORY
    # =====================================================

    user_history = df[
        df['user_id'] == user_id
    ][
        ['item_id', 'rating']
    ]

    watched_movies = set(
        user_history['item_id']
    )

    history_items = []
    history_ratings = []

    for _, row in user_history.iterrows():

        item_id = row['item_id']

        if item_id not in item_to_index:
            continue

        history_items.append(
            item_to_index[item_id]
        )

        history_ratings.append(
            row['rating']
        )

    if len(history_items) == 0:
        print("No valid history found.")
        return []

    # =====================================================
    # BUILD USER EMBEDDING
    # =====================================================

    history_items_tensor = torch.tensor(
        [history_items],
        dtype=torch.long
    ).to(DEVICE)

    history_ratings_tensor = torch.tensor(
        [history_ratings],
        dtype=torch.float32
    ).to(DEVICE)

    with torch.no_grad():

        user_embedding, _ = model.build_user_embedding(
            history_items_tensor,
            history_ratings_tensor
        )

    # =====================================================
    # CANDIDATE MOVIES
    # =====================================================

    candidate_movies = [
        item_id
        for item_id in item_to_index.keys()
        if item_id not in watched_movies
    ]

    predictions = []

    # =====================================================
    # PREDICT
    # =====================================================

    with torch.no_grad():

        for item_id in candidate_movies:

            item_idx = item_to_index[item_id]

            item_tensor = torch.tensor(
                [item_idx],
                dtype=torch.long
            ).to(DEVICE)

            prediction = model(
                user_embedding,
                item_tensor
            )

            predictions.append(
                (
                    item_id,
                    prediction.item()
                )
            )

    # =====================================================
    # SORT
    # =====================================================

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_k]

In [44]:
sample_user = df['user_id'].iloc[10]

recommendations = recommend_movies(
    sample_user,
    top_k=10
)

recommendations

[(np.int64(143842), 4.923251152038574),
 (np.int64(282051), 4.903809547424316),
 (np.int64(109669), 4.900166034698486),
 (np.int64(165150), 4.900115966796875),
 (np.int64(295746), 4.891871929168701),
 (np.int64(158429), 4.88456916809082),
 (np.int64(6094), 4.8813934326171875),
 (np.int64(146948), 4.873692989349365),
 (np.int64(207230), 4.873377799987793),
 (np.int64(20286), 4.873228073120117)]

In [ ]:
# for epoch in range(EPOCHS):

#     # =========================
#     # TRAINING
#     # =========================

#     model.train()

#     train_loss = 0

#     for (

#         history_items,
#         history_ratings,
#         target_items,
#         labels

#     ) in tqdm(

#         train_loader,
#         desc=f"Training Epoch {epoch+1}"

#     ):

#         history_items = history_items.to(DEVICE)

#         history_ratings = history_ratings.to(DEVICE)

#         target_items = target_items.to(DEVICE)

#         labels = labels.to(DEVICE)


#         predictions = model(

#             history_items,
#             history_ratings,
#             target_items

#         )


#         loss = criterion(
#             predictions,
#             labels
#         )


#         # Embedding regularization
#         reg_loss = (
#             model.item_embedding.weight
#             .norm(2)
#             .pow(2)
#         )

#         loss = (
#             loss
#             + lambda_reg * reg_loss
#         )


#         optimizer.zero_grad()

#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(
#             model.parameters(),
#             max_norm=1.0
#         )

#         optimizer.step()

#         train_loss += loss.item()


#     avg_train_loss = (
#         train_loss / len(train_loader)
#     )


#     # =========================
#     # VALIDATION
#     # =========================

#     model.eval()

#     val_loss = 0

#     with torch.no_grad():

#         for (

#             history_items,
#             history_ratings,
#             target_items,
#             labels

#         ) in tqdm(

#             val_loader,
#             desc=f"Validation Epoch {epoch+1}"

#         ):

#             history_items = history_items.to(DEVICE)

#             history_ratings = history_ratings.to(DEVICE)

#             target_items = target_items.to(DEVICE)

#             labels = labels.to(DEVICE)


#             predictions = model(

#                 history_items,
#                 history_ratings,
#                 target_items

#             )


#             loss = criterion(
#                 predictions,
#                 labels
#             )


#             reg_loss = (
#                 model.item_embedding.weight
#                 .norm(2)
#                 .pow(2)
#             )

#             loss = (
#                 loss
#                 + lambda_reg * reg_loss
#             )

#             val_loss += loss.item()


#     avg_val_loss = (
#         val_loss / len(val_loader)
#     )


#     # =========================
#     # LOGGING
#     # =========================

#     print(

#         f"Epoch {epoch+1} | "
#         f"Train Loss: {avg_train_loss:.4f} | "
#         f"Val Loss: {avg_val_loss:.4f}"

#     )


#     # =========================
#     # EARLY STOPPING
#     # =========================

#     if avg_val_loss < best_val_loss:

#         best_val_loss = avg_val_loss

#         patience_counter = 0

#         torch.save(
#             model.state_dict(),
#             "best_dynamic_ncf_dynamic.pth"
#         )

#         print(
#             "Validation improved → model saved"
#         )

#     else:

#         patience_counter += 1

#         print(
#             f"No improvement "
#             f"({patience_counter}/{PATIENCE})"
#         )

#         if patience_counter >= PATIENCE:

#             print(
#                 "Early stopping triggered"
#             )

#             break

In [ ]:
for epoch in range(EPOCHS):

    # =========================
    # TRAINING
    # =========================

    model.train()

    train_loss = 0

    for (

        history_items,
        history_ratings,
        target_items,
        labels

    ) in tqdm(

        train_loader,
        desc=f"Training Epoch {epoch+1}"

    ):

        history_items = history_items.to(DEVICE)

        history_ratings = history_ratings.to(DEVICE)

        target_items = target_items.to(DEVICE)

        labels = labels.to(DEVICE)


        # ====================================
        # BUILD DYNAMIC USER EMBEDDINGS
        # ====================================

        history_embeds = model.item_embedding(
            history_items
        )

        history_ratings_expanded = (
            history_ratings.unsqueeze(-1)
        )

        weighted_history = (
            history_embeds
            * history_ratings_expanded
        )


        # ====================================
        # MASK PADDING
        # ====================================

        mask = (
            history_items != 0
        ).unsqueeze(-1)


        weighted_history = (
            weighted_history * mask
        )


        history_lengths = (
            mask.sum(dim=1)
            .clamp(min=1)
        )


        # ====================================
        # DYNAMIC USER EMBEDDING
        # ====================================

        user_embedding = (

            weighted_history.sum(dim=1)

            / history_lengths

        )


        # ====================================
        # FORWARD PASS
        # ====================================

        predictions = model(

            user_embedding,
            target_items

        )


        # ====================================
        # LOSS
        # ====================================

        loss = criterion(
            predictions,
            labels
        )


        # ====================================
        # REGULARIZATION
        # ====================================

        reg_loss = (
            model.item_embedding.weight
            .norm(2)
            .pow(2)
        )


        loss = (
            loss
            + lambda_reg * reg_loss
        )


        # ====================================
        # BACKPROP
        # ====================================

        optimizer.zero_grad()

        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()

        train_loss += loss.item()


    avg_train_loss = (
        train_loss / len(train_loader)
    )


    # ====================================
    # VALIDATION
    # ====================================

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for (

            history_items,
            history_ratings,
            target_items,
            labels

        ) in tqdm(

            val_loader,
            desc=f"Validation Epoch {epoch+1}"

        ):

            history_items = history_items.to(DEVICE)

            history_ratings = history_ratings.to(DEVICE)

            target_items = target_items.to(DEVICE)

            labels = labels.to(DEVICE)


            # ================================
            # BUILD USER EMBEDDINGS
            # ================================

            history_embeds = (
                model.item_embedding(
                    history_items
                )
            )


            history_ratings_expanded = (
                history_ratings.unsqueeze(-1)
            )


            weighted_history = (

                history_embeds
                * history_ratings_expanded

            )


            mask = (
                history_items != 0
            ).unsqueeze(-1)


            weighted_history = (
                weighted_history * mask
            )


            history_lengths = (
                mask.sum(dim=1)
                .clamp(min=1)
            )


            user_embedding = (

                weighted_history.sum(dim=1)

                / history_lengths

            )


            # ================================
            # PREDICTION
            # ================================

            predictions = model(

                user_embedding,
                target_items

            )


            # ================================
            # LOSS
            # ================================

            loss = criterion(
                predictions,
                labels
            )


            reg_loss = (
                model.item_embedding.weight
                .norm(2)
                .pow(2)
            )


            loss = (
                loss
                + lambda_reg * reg_loss
            )

            val_loss += loss.item()


    avg_val_loss = (
        val_loss / len(val_loader)
    )


    # ====================================
    # LOGGING
    # ====================================

    print(

        f"Epoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"

    )


    # ====================================
    # EARLY STOPPING
    # ====================================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        patience_counter = 0

        torch.save(

            model.state_dict(),

            "best_dynamic_ncf_online.pth"

        )

        print(
            "Validation improved → model saved"
        )

    else:

        patience_counter += 1

        print(

            f"No improvement "
            f"({patience_counter}/{PATIENCE})"

        )

        if patience_counter >= PATIENCE:

            print(
                "Early stopping triggered"
            )

            break

### Static Embedding

In [ ]:
class RatingsDataset(Dataset):

    def __init__(self, dataframe):

        self.users = torch.tensor(
            dataframe['user_idx'].values,
            dtype=torch.long
        )

        self.items = torch.tensor(
            dataframe['item_idx'].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            dataframe['normalized_rating'].values,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.ratings)

    def __getitem__(self, idx):

        return (
            self.users[idx],
            self.items[idx],
            self.ratings[idx]
        )

In [ ]:
train_dataset = RatingsDataset(train_df)
test_dataset = RatingsDataset(test_df)
val_dataset = RatingsDataset(val_df)


train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=2048,
    shuffle=False
)


val_loader = DataLoader(
    val_dataset,
    batch_size=2048,
    shuffle=False
)

### Collaborative Filtering Model

In [ ]:
class CollaborativeFilteringModel(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=50
    ):

        super().__init__()

        # User embeddings
        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        # Item embeddings
        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        # User bias
        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        # Item bias
        self.item_bias = nn.Embedding(
            num_items,
            1
        )

    def forward(self, user_ids, item_ids):

        # Embedding lookup
        user_vecs = self.user_embedding(user_ids)
        item_vecs = self.item_embedding(item_ids)

        # Dot product
        interaction = (
            user_vecs * item_vecs
        ).sum(dim=1)

        # Biases
        user_b = self.user_bias(user_ids).squeeze()
        item_b = self.item_bias(item_ids).squeeze()

        prediction = (
            interaction
            + user_b
            + item_b
        )

        final_prediction = (
            prediction
            + item_mean
        )

        return final_prediction

In [ ]:
class NeuralCollaborativeFiltering(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=32
    ):

        super().__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        # Biases
        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.item_bias = nn.Embedding(
            num_items,
            1
        )

        # Neural interaction layers
        self.mlp = nn.Sequential(

            nn.Linear(
                embedding_dim * 2,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                64,
                1
            )
        )

        # Global bias
        self.global_bias = nn.Parameter(
            torch.tensor([0.0])
        )

        # Initialization
        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.item_embedding.weight,
            std=0.01
        )

    def forward(
        self,
        user_ids,
        item_ids
    ):

        # Embeddings
        user_vecs = self.user_embedding(
            user_ids
        )

        item_vecs = self.item_embedding(
            item_ids
        )

        # Concatenate
        x = torch.cat(
            [user_vecs, item_vecs],
            dim=1
        )

        # Neural interaction
        interaction = self.mlp(x).squeeze()

        # Bias terms
        user_b = self.user_bias(
            user_ids
        ).squeeze()

        item_b = self.item_bias(
            item_ids
        ).squeeze()

        prediction = (
            self.global_bias
            + interaction
            + user_b
            + item_b
        )

        return prediction

In [ ]:
num_users = len(user_to_index)
num_items = len(item_to_index)


model = NeuralCollaborativeFiltering(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=100
)

print(model)

### Training Setup

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

model = model.to(DEVICE)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

### Training

In [ ]:
EPOCHS = 50

PATIENCE = 3

best_val_loss = float('inf')

patience_counter = 0

lambda_reg = 1e-5

for epoch in range(EPOCHS):

    # =========================
    # TRAINING
    # =========================

    model.train()

    train_loss = 0

    for users, items, ratings in tqdm(
        train_loader,
        desc=f"Training Epoch {epoch+1}"
    ):

        users = users.to(DEVICE)
        items = items.to(DEVICE)
        ratings = ratings.to(DEVICE)

        predictions = model(
            users,
            items
        )

        loss = criterion(
            predictions,
            ratings
        )

        reg_loss = (
            model.user_embedding.weight.norm(2).pow(2)
            +
            model.item_embedding.weight.norm(2).pow(2)
        )
        
        loss = (
            loss
            + lambda_reg * reg_loss
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = (
        train_loss / len(train_loader)
    )


    # =========================
    # VALIDATION
    # =========================

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for users, items, ratings in tqdm(
            val_loader,
            desc=f"Validation Epoch {epoch+1}"
        ):

            users = users.to(DEVICE)
            items = items.to(DEVICE)
            ratings = ratings.to(DEVICE)

            predictions = model(
                users,
                items
            )

            loss = criterion(
                predictions,
                ratings
            )

            reg_loss = (
                model.user_embedding.weight.norm(2).pow(2)
                +
                model.item_embedding.weight.norm(2).pow(2)
            )
            
            loss = (
                loss
                + lambda_reg * reg_loss
            )

            val_loss += loss.item()

    avg_val_loss = (
        val_loss / len(val_loader)
    )


    # =========================
    # LOGGING
    # =========================

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )


    # =========================
    # EARLY STOPPING
    # =========================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        patience_counter = 0

        # Save best model
        torch.save(
            model.state_dict(),
            "best_model_ncf.pth"
        )

        print("Validation improved → model saved")

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print("Early stopping triggered")

            break

In [ ]:
EPOCHS = 30


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for users, items, ratings in tqdm(train_loader):

        users = users.to(DEVICE)
        items = items.to(DEVICE)
        ratings = ratings.to(DEVICE)

        predictions = model(
            users,
            items
        )

        loss = criterion(
            predictions,
            ratings
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} | Loss: {avg_loss:.4f}"
    )

In [ ]:
model.load_state_dict(
    torch.load(
        "best_model_ncf.pth",
        map_location=DEVICE
    )
)

model.eval()

print("Best model loaded.")

In [ ]:
from sklearn.metrics import mean_squared_error

model.eval()

all_predictions = []
all_targets = []

test_loss = 0


with torch.no_grad():

    for users, items, ratings in tqdm(test_loader):

        users = users.to(
            DEVICE,
            non_blocking=True
        )

        items = items.to(
            DEVICE,
            non_blocking=True
        )

        ratings = ratings.to(
            DEVICE,
            non_blocking=True
        )

        predictions = model(
            users,
            items
        )

        loss = criterion(
            predictions,
            ratings
        )

        test_loss += loss.item()

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            ratings.cpu().numpy()
        )


avg_test_loss = (
    test_loss / len(test_loader)
)

rmse = np.sqrt(
    mean_squared_error(
        all_targets,
        all_predictions
    )
)

print(f"Test Loss : {avg_test_loss:.4f}")
print(f"Test RMSE : {rmse:.4f}")

In [ ]:
from sklearn.metrics import mean_squared_error

model.eval()

all_predictions = []
all_targets = []

test_loss = 0


with torch.no_grad():

    for (

        history_items,
        history_ratings,
        target_items,
        labels

    ) in tqdm(

        test_loader,
        desc="Testing"

    ):

        history_items = history_items.to(
            DEVICE,
            non_blocking=True
        )

        history_ratings = history_ratings.to(
            DEVICE,
            non_blocking=True
        )

        target_items = target_items.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        # ====================================
        # BUILD DYNAMIC USER EMBEDDINGS
        # ====================================

        history_embeds = model.item_embedding(
            history_items
        )


        history_ratings_expanded = (
            history_ratings.unsqueeze(-1)
        )


        weighted_history = (

            history_embeds
            * history_ratings_expanded

        )


        # ====================================
        # MASK PADDING
        # ====================================

        mask = (
            history_items != 0
        ).unsqueeze(-1)


        weighted_history = (
            weighted_history * mask
        )


        history_lengths = (
            mask.sum(dim=1)
            .clamp(min=1)
        )


        user_embedding = (

            weighted_history.sum(dim=1)

            / history_lengths

        )


        # ====================================
        # PREDICTION
        # ====================================

        predictions = model(

            user_embedding,
            target_items

        )


        # ====================================
        # LOSS
        # ====================================

        loss = criterion(
            predictions,
            labels
        )

        test_loss += loss.item()


        # ====================================
        # STORE RESULTS
        # ====================================

        all_predictions.extend(

            predictions.cpu().numpy()

        )

        all_targets.extend(

            labels.cpu().numpy()

        )


avg_test_loss = (
    test_loss / len(test_loader)
)


rmse = np.sqrt(

    mean_squared_error(

        all_targets,
        all_predictions

    )

)

print(f"Test Loss : {avg_test_loss:.4f}")

print(f"Test RMSE : {rmse:.4f}")

In [ ]:
def recommend_movies(

    user_id,
    top_k=10

):

    model.eval()


    # ====================================
    # USER HISTORY
    # ====================================

    user_history = user_histories[
        user_to_index[user_id]
    ]


    history_items = [
        x[0]
        for x in user_history
    ]


    history_ratings = [
        x[1]
        for x in user_history
    ]


    history_items_tensor = torch.tensor(

        history_items,
        dtype=torch.long

    ).unsqueeze(0).to(DEVICE)


    history_ratings_tensor = torch.tensor(

        history_ratings,
        dtype=torch.float32

    ).unsqueeze(0).to(DEVICE)


    # ====================================
    # BUILD USER EMBEDDING
    # ====================================

    with torch.no_grad():

        history_embeds = model.item_embedding(
            history_items_tensor
        )


        history_ratings_expanded = (
            history_ratings_tensor.unsqueeze(-1)
        )


        weighted_history = (

            history_embeds
            * history_ratings_expanded

        )


        mask = (
            history_items_tensor != 0
        ).unsqueeze(-1)


        weighted_history *= mask


        history_lengths = (
            mask.sum(dim=1)
            .clamp(min=1)
        )


        user_embedding = (

            weighted_history.sum(dim=1)

            / history_lengths

        )


        # ====================================
        # CANDIDATE MOVIES
        # ====================================

        watched_movies = set(
            history_items
        )


        candidate_indices = [

            idx

            for idx in range(1, num_items)

            if idx not in watched_movies

        ]


        candidate_tensor = torch.tensor(

            candidate_indices,
            dtype=torch.long

        ).to(DEVICE)


        # ====================================
        # REPEAT USER EMBEDDING
        # ====================================

        repeated_user_embedding = (

            user_embedding.repeat(
                len(candidate_indices),
                1
            )

        )


        # ====================================
        # PREDICT RATINGS
        # ====================================

        predictions = model(

            repeated_user_embedding,
            candidate_tensor

        )


        # ====================================
        # ADD ITEM MEANS BACK
        # ====================================

        item_means = torch.tensor(

            [
                item_mean_ratings[idx]
                for idx in candidate_indices
            ],

            dtype=torch.float32

        ).to(DEVICE)


        predictions = (
            predictions
            + item_means
        )

        # predictions = torch.clamp(
        #     predictions,
        #     min=0.5,
        #     max=5.0
        # )


        # ====================================
        # TOP-K
        # ====================================

        top_scores, top_positions = (
            torch.topk(
                predictions,
                k=top_k,
                largest=True
            )
        )


    recommendations = []


    for score, position in zip(

        top_scores.cpu().numpy(),
        top_positions.cpu().numpy()

    ):

        item_idx = candidate_indices[position]

        item_id = index_to_item[item_idx]

        recommendations.append(

            (
                item_id,
                score
            )

        )


    return recommendations

In [ ]:
sample_user = df['user_id'].iloc[5]

recommendations = recommend_movies(
    sample_user,
    top_k=10
)

recommendations

In [31]:
model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for users, items, ratings in test_loader:

        users = users.to(DEVICE)
        items = items.to(DEVICE)

        predictions = model(
            users,
            items
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            ratings.numpy()
        )


rmse = np.sqrt(
    mean_squared_error(
        all_targets,
        all_predictions
    )
)

print(f"RMSE: {rmse:.4f}")

ValueError: too many values to unpack (expected 3)

In [29]:
def recommend_movies(
    user_id,
    top_k=10
):

    model.eval()

    user_idx = user_to_index[user_id]

    watched_movies = set(
        df[
            df['user_id'] == user_id
        ]['item_id']
    )

    candidate_movies = [
        item_id
        for item_id in item_to_index.keys()
        if item_id not in watched_movies
    ]

    predictions = []

    with torch.no_grad():

        for item_id in candidate_movies:

            item_idx = item_to_index[item_id]

            user_tensor = torch.tensor(
                [user_idx],
                dtype=torch.long
            ).to(DEVICE)

            item_tensor = torch.tensor(
                [item_idx],
                dtype=torch.long
            ).to(DEVICE)

            prediction = model(
                user_tensor,
                item_tensor
            )

            predictions.append(
                (
                    item_id,
                    prediction.item()
                )
            )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_k]

In [30]:
sample_user = df['user_id'].iloc[0]

recommendations = recommend_movies(
    sample_user,
    top_k=10
)

recommendations

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)